# **TCN**

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
from preprocessing.ForTorch import set_seed, DEVICE, SeqDataset, torch_predict, train_torch_classifier, train_tabnet_ttp
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

TCN — это Temporal Convolutional Network, по-русски: сверточная сеть для временных рядов.
По сути это альтернатива RNN/LSTM/GRU, которая работает не через рекурсию, а через 1D-свёртки, но специально сконструированные под “только прошлое → будущее”.

In [ ]:
class Chomp1d(nn.Module):
    def init(self, chomp):
        super().init()
        self.chomp = chomp
    def forward(self, x):
        return x[:, :, :-self.chomp] if self.chomp > 0 else x

class TemporalBlock(nn.Module):
    def init(self, in_ch, out_ch, k=3, d=1, dropout=0.2):
        super().init()
        pad = (k-1)*d
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k, padding=pad, dilation=d),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, k, padding=pad, dilation=d),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        y = self.net(x)
        return self.relu(y + self.down(x))

class TCNCls(nn.Module):
    def init(self, n_features, channels=(64,64,64), n_classes=3, dropout=0.2):
        super().init()
        layers = []
        in_ch = n_features
        for i, ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, ch, k=3, d=2**i, dropout=dropout))
            in_ch = ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(in_ch, n_classes)

    def forward(self, x):        # (B,L,F)
        x = x.transpose(1,2)     # (B,F,L)
        y = self.tcn(x)          # (B,C,L)
        h = y[:, :, -1]          # last time
        return self.fc(h)

def train_tcn_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False)

        model = TCNCls(n_features=X_train.shape[1], channels=(64,64,64), n_classes=3)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=20, lr=1e-3)
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"TCN",
            "model_family":"rnn",
            "model_params":{"channels":[64,64,64],"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })

In [ ]:
class Chomp1d(nn.Module):
    def init(self, chomp):
        super().init()
        self.chomp = chomp
    def forward(self, x):
        return x[:, :, :-self.chomp] if self.chomp > 0 else x

class TemporalBlock(nn.Module):
    def init(self, in_ch, out_ch, k=3, d=1, dropout=0.2):
        super().init()
        pad = (k-1)*d
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k, padding=pad, dilation=d),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, k, padding=pad, dilation=d),
            Chomp1d(pad),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU()

    def forward(self, x):
        y = self.net(x)
        return self.relu(y + self.down(x))

class TCNCls(nn.Module):
    def init(self, n_features, channels=(64,64,64), n_classes=3, dropout=0.2):
        super().init()
        layers = []
        in_ch = n_features
        for i, ch in enumerate(channels):
            layers.append(TemporalBlock(in_ch, ch, k=3, d=2**i, dropout=dropout))
            in_ch = ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(in_ch, n_classes)

    def forward(self, x):        # (B,L,F)
        x = x.transpose(1,2)     # (B,F,L)
        y = self.tcn(x)          # (B,C,L)
        h = y[:, :, -1]          # last time
        return self.fc(h)

def train_tcn_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False)

        model = TCNCls(n_features=X_train.shape[1], channels=(64,64,64), n_classes=3)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=20, lr=1e-3)
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"TCN",
            "model_family":"rnn",
            "model_params":{"channels":[64,64,64],"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })

In [ ]:
train_tcn_ttp(df, 1000, 200, 100, seq_len=64)